# 阶段 3：论文原版 1.5B 模型的基线（Colab）

原作者给的三个 1.5B 模型（sft / grpo / score-grpo）用**论文格式**（长提示词 + 中文动作名）在**和我们的模型同一批测试状态**上各答一遍，
答案写成和训练 notebook 一样的 `preds.csv`，Mac 上 `python -m eval.open_loop` 用同一套指标打分。

不用改任何东西，直接「代码执行程序 → 全部运行」。

| 步骤 | 做什么 | G4 上预计 |
|---|---|---|
| 拼法试探 | sft 模型在 60 条上试 5 种拼法，挑和原作者标签一致最多的。第一种 `repo` 是原作者仓库训练和评测脚本里的拼法，应该是它胜出 | 约 3 分钟（估算） |
| 三个模型各答一遍 | 原作者干净测试集 245 条 + 我们的两份测试集 | 每个模型约 3 分钟（11/4 实测 3.1 分钟） |

跑之前：运行时选 **G4**（没有就 L4 或 A100）；Drive 里要有 `MyDrive/COMLLM/s3_data/`（训练 notebook 用的同一份）和 `MyDrive/COMLLM/reference_model/` 下的三个模型文件夹。

**结果**在 Drive 的 `MyDrive/COMLLM/s3_runs/paper_<UTC 时间>/`，里面每个模型一个子文件夹。

**止损线**：
- 第一个模型答了 5 批后会打印预计时长。单个模型超过 40 分钟就停下来告诉我，把第 3 格的 `N_PER_COUNT` 改成 200 再跑。
- 第 6 格如果打印「⚠ 拼法可能不对」，先别往下跑，把第 6 格的输出截图发到对话里。
- 任何一格报错就停，截图发到对话里。
- 跑完「代码执行程序 → 断开连接并删除运行时」。

In [ ]:
# 第 1 格：安装固定版本的库（版本固定，以后复现时环境一样）
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.20.2", "peft==0.21.2", "accelerate", "sentencepiece"], check=True)
    # Colab 预装的 torchao 版本太老，peft 0.21 检测到它会直接报错。这里用不到 torchao，卸掉
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

import importlib, importlib.util
importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None, "torchao 还在：菜单「代码执行程序 → 重新启动会话」后再「全部运行」"
print("依赖安装完成")

In [ ]:
# 第 2 格：检查 GPU。L4 / A100 支持 bf16；T4 不支持，这个 notebook 不在 T4 上跑
import os, time, json, math, hashlib, csv, shutil
from datetime import datetime, timezone
import numpy as np
import torch

if IN_COLAB:
    subprocess.run(["nvidia-smi"], check=False)
    assert torch.cuda.is_available(), "没检测到 GPU：菜单「代码执行程序 → 更改运行时类型」选 L4 GPU 后再运行"
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
if HAS_GPU:
    assert torch.cuda.is_bf16_supported(), "这块 GPU 不支持 bf16（T4 就不支持）：换成 L4"
DTYPE = torch.bfloat16 if HAS_GPU else torch.float32
print("GPU:", torch.cuda.get_device_name() if HAS_GPU else "无（CPU）", "| 精度:", "bf16" if HAS_GPU else "fp32")
print("torch", torch.__version__, "（第一次跑时把这个版本号记进 docs/versions.md）")

In [ ]:
# 第 3 格：设置（一般不用改）
MODELS_TO_RUN = ["sft", "grpo", "score-grpo"]   # 时间不够就先只留 "sft"
N_PER_COUNT = None      # None = 测试集全部；太慢就改成 200（每种服务器台数只取前 200 条）
FORMAT = None           # None = 先自动试探拼法；已经知道就填 "repo" / "json" / "json_ascii" / "repr" / "sys_instr"

# ---- 下面不用改 ----
REF_NAME = "qwen2.5-1.5b-mec-{}-final-all"     # Drive 上三个模型文件夹的名字
MAX_NEW_TOKENS = 20     # "卸载到边缘服务器11" 约 8 个 token，留余量
BATCH = 32 if HAS_GPU else 4
N_PROBE = 60            # 试探拼法用的条数
MAX_SAME_PCT = 90.0     # 试探时胜出的拼法有超过这个比例的回答是同一个动作 → 提醒拼法可能不对

# 本机测试用（Colab 上这些环境变量都不存在，不起作用）
LOCAL_MODEL = os.environ.get("S3_LOCAL_MODEL")
if os.environ.get("S3P_N_PER_COUNT"):
    N_PER_COUNT = int(os.environ["S3P_N_PER_COUNT"])
if os.environ.get("S3P_MODELS"):
    MODELS_TO_RUN = os.environ["S3P_MODELS"].split(",")
if os.environ.get("S3P_FORMAT"):
    FORMAT = os.environ["S3P_FORMAT"]

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/COMLLM/s3_data"
    REF_DIR = "/content/drive/MyDrive/COMLLM/reference_model"
    OUT = f"/content/drive/MyDrive/COMLLM/s3_runs/paper_{STAMP}"
else:
    DATA_DIR = os.environ["S3_DATA"]
    REF_DIR = None
    OUT = os.path.join(os.environ["S3_OUT"], f"paper_{STAMP}")
os.makedirs(OUT, exist_ok=True)


def model_dir(m):
    if LOCAL_MODEL:
        return LOCAL_MODEL
    return os.path.join(REF_DIR, REF_NAME.format(m))


for m in MODELS_TO_RUN:
    assert os.path.exists(os.path.join(model_dir(m), "config.json")), f"找不到模型文件夹 {model_dir(m)}"
print("要跑的模型:", MODELS_TO_RUN, "| 每种台数条数:", N_PER_COUNT or "全部")
print("结果目录:", OUT)

In [ ]:
# 第 4 格：读三份测试集（和训练 notebook 同一份文件），核对校验值
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


with open(os.path.join(DATA_DIR, "meta.json"), encoding="utf-8") as f:
    META = json.load(f)
SPLITS = [s for s in ("friend_clean", "test_seen", "test_unseen") if s in META["files"]]
DATA = {}
for split in SPLITS:
    path = os.path.join(DATA_DIR, split + ".jsonl")
    assert md5_of(path) == META["files"][split]["md5"], f"{split}.jsonl 的校验值和 meta.json 对不上"
    DATA[split] = read_jsonl(path)
    for i, r in enumerate(DATA[split]):
        r["_idx"] = i

# 要答的行：原作者干净测试集全部；我们的测试集按 N_PER_COUNT 截取
ROWS = []
for split in SPLITS:
    count = {}
    for r in DATA[split]:
        k = r["servers"]
        count[k] = count.get(k, 0) + 1
        if split == "friend_clean" or N_PER_COUNT is None or count[k] <= N_PER_COUNT:
            ROWS.append(r)
for split in SPLITS:
    n = 0
    for r in ROWS:
        if r["split"] == split:
            n += 1
    print(f"{split:13s} 答 {n:5d} 条（文件里共 {len(DATA[split])} 条）")
print("合计", len(ROWS), "条")

INSTR = META["paper_instruction"]
QWEN_SYSTEM = META["qwen_default_system"]
REPO_SYSTEM = "你是一个MEC任务卸载专家。请严格按照用户的指示分析输入数据，输出卸载策略。不要包含任何解释、代码块标记或额外文本。"
INPUT_MARKER = "**输入数据**："


def chatml(messages):
    """和仓库里 deploy/prompts.py 的 chatml 一模一样。"""
    text = ""
    for m in messages:
        text = text + "<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>\n"
    text = text + "<|im_start|>assistant\n"
    return text


def mec_lines(inp):
    """状态字典 → 每行一个 "键: 值" 的文字；值是列表或字典时写成 JSON。

    1. 按字典里的顺序逐个取出键和值
    2. 值是列表或字典 → json.dumps（中文原样）；否则直接转成文字
    3. 每对拼成 "键: 值"，用换行连起来
    """
    lines = []
    for key in inp:
        value = inp[key]
        if isinstance(value, (list, dict)):
            value = json.dumps(value, ensure_ascii=False)
        lines.append(key + ": " + str(value))
    return "\n".join(lines)


def paper_messages(row, fmt):
    """论文格式的对话。论文没写用户内容怎么拼，这里列 5 种：

    repo       原作者仓库的训练和评测脚本用的拼法：系统提示 = "你是一个MEC任务卸载专家……"；
               用户 = 说明 + 空一行 + "**输入数据**：" + 换行 + 每行一个 "键: 值"
    json       系统提示 = Qwen 默认那句；用户 = 说明 + 换行 + 状态字典的 JSON（中文原样）。阶段 2 量延迟用的是这种
    json_ascii 同上，但 JSON 里的中文转成 \\uXXXX（json.dumps 不加 ensure_ascii=False 时的样子）
    repr       同上，但状态字典用 Python 的 str() 转文字（单引号）
    sys_instr  系统提示 = 说明；用户 = 状态字典的 JSON
    """
    inp = row["paper_input"]
    if fmt == "repo":
        return [{"role": "system", "content": REPO_SYSTEM},
                {"role": "user", "content": INSTR + "\n\n" + INPUT_MARKER + "\n" + mec_lines(inp)}]
    if fmt == "json":
        return [{"role": "system", "content": QWEN_SYSTEM},
                {"role": "user", "content": INSTR + "\n" + json.dumps(inp, ensure_ascii=False)}]
    if fmt == "json_ascii":
        return [{"role": "system", "content": QWEN_SYSTEM},
                {"role": "user", "content": INSTR + "\n" + json.dumps(inp)}]
    if fmt == "repr":
        return [{"role": "system", "content": QWEN_SYSTEM},
                {"role": "user", "content": INSTR + "\n" + str(inp)}]
    if fmt == "sys_instr":
        return [{"role": "system", "content": INSTR},
                {"role": "user", "content": json.dumps(inp, ensure_ascii=False)}]
    raise ValueError(fmt)


FORMATS = ("repo", "json", "json_ascii", "repr", "sys_instr")

# 核对：json 拼法 = Mac 上仓库代码拼的（meta.json 里的样例）
for i, golden in enumerate(META["golden"]["paper"]):
    assert chatml(paper_messages(DATA["test_seen"][i], "json")) == golden, "论文格式提示词和 Mac 上拼的不一样"
print("论文格式提示词核对通过 ✓")

In [ ]:
# 第 5 格：生成和解析答案的函数（这一格只定义函数，不跑东西）
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

ACTION_RE = re.compile(r"本地执行|卸载到边缘服务器(\d+)")    # 和 deploy/prompts.py 的 PAPER_ACTION_RE 一样


def parse_action(text, n_servers):
    """生成的文字 → 动作编号：本地 = 0，服务器 k = k；认不出来或服务器编号超出台数 → −1（非法）。

    1. 在文字里找第一个"本地执行"或"卸载到边缘服务器k"
    2. 找不到 → −1
    3. 服务器编号要在 1 到 n_servers 之间，否则 → −1
    """
    m = ACTION_RE.search(text)
    if m is None:
        return -1
    if m.group(0) == "本地执行":
        return 0
    k = int(m.group(1))
    if 1 <= k <= n_servers:
        return k
    return -1


assert parse_action("卸载到边缘服务器3", 5) == 3
assert parse_action("本地执行", 5) == 0
assert parse_action("卸载到边缘服务器7", 5) == -1
assert parse_action("我觉得应该卸载到边缘服务器11。", 11) == 11
assert parse_action("卸载到服务器2", 5) == -1


def load_model(m):
    """从 Drive 读一个原作者模型和它自己的分词器。grpo 两个存的是 fp16，读进来统一转成 bf16。"""
    d = model_dir(m)
    t0 = time.time()
    tk = AutoTokenizer.from_pretrained(d)
    tk.padding_side = "left"
    if tk.pad_token is None:
        tk.pad_token = "<|endoftext|>"
    md = AutoModelForCausalLM.from_pretrained(d, dtype=DTYPE).to(DEVICE)
    md.eval()
    print(f"读入 {m}：{time.time() - t0:.0f} 秒")
    return tk, md


@torch.no_grad()
def generate(tk, md, rows, fmt, show_eta=False):
    """每行生成一段回答，返回 {(split, 行号): 生成的文字}。

    1. 每行按 fmt 拼好提示词全文（模型自己的对话模板），切成 token
    2. 按服务器台数分组（同组提示词长度几乎一样），每组切成 BATCH 条一批
    3. 左补齐，贪心解码（每步取概率最大的 token），最多 MAX_NEW_TOKENS 个，遇到 <|im_end|> 停
    4. 只把新生成的部分解码成文字
    """
    im_end = tk.convert_tokens_to_ids("<|im_end|>")
    gen_cfg = GenerationConfig(max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                               pad_token_id=tk.pad_token_id, eos_token_id=[im_end, tk.eos_token_id])
    groups = {}
    for r in rows:
        groups.setdefault(r["servers"], []).append(r)
    batches = []
    for k in sorted(groups):
        g = groups[k]
        for s in range(0, len(g), BATCH):
            batches.append(g[s:s + BATCH])
    out = {}
    t0 = time.time()
    for b, batch in enumerate(batches):
        texts = []
        for r in batch:
            texts.append(tk.apply_chat_template(paper_messages(r, fmt), tokenize=False, add_generation_prompt=True))
        enc = tk(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        gen = md.generate(**enc, generation_config=gen_cfg)
        new = gen[:, enc["input_ids"].shape[1]:]
        for i, r in enumerate(batch):
            out[(r["split"], r["_idx"])] = tk.decode(new[i], skip_special_tokens=True).strip()
        if show_eta and b == 4:
            per = (time.time() - t0) / 5
            print(f"  前 5 批用了 {time.time() - t0:.0f} 秒，这个模型预计共约 {per * len(batches) / 60:.0f} 分钟（估算）")
    return out


def check_template(tk):
    """模型自带的对话模板拼出来的，要和 chatml 拼的一样（不一样只提醒，不停下：以模型自带的为准）。"""
    r = DATA["test_seen"][0]
    for fmt in FORMATS:
        hf = tk.apply_chat_template(paper_messages(r, fmt), tokenize=False, add_generation_prompt=True)
        if hf != chatml(paper_messages(r, fmt)):
            print(f"⚠ 拼法 {fmt}：模型自带的对话模板和 chatml 不一样，用模型自带的")

In [ ]:
# 第 6 格：试探拼法。sft 模型在 60 条原作者测试样本上用 5 种拼法各答一遍，
#        和原作者的标签比，答对最多的就是他训练时用的拼法（并列时选非法少的）。
#        还记下每种拼法里最常见的那个动作占多少：拼法不对时模型往往不管状态都答同一个动作
probe_rows = DATA["friend_clean"][::4][:N_PROBE]
probe = {}
if FORMAT is None:
    tk, md = load_model(MODELS_TO_RUN[0])
    check_template(tk)
    best_fmt = None
    for fmt in FORMATS:
        texts = generate(tk, md, probe_rows, fmt)
        hit = 0
        invalid = 0
        dist = {}
        for r in probe_rows:
            a = parse_action(texts[(r["split"], r["_idx"])], r["servers"])
            dist[a] = dist.get(a, 0) + 1
            if a < 0:
                invalid += 1
            elif a == r["friend_label"]:
                hit += 1
        top = 0
        for a in dist:
            if dist[a] > top:
                top = dist[a]
        probe[fmt] = {"agree_with_label_pct": round(100.0 * hit / len(probe_rows), 1),
                      "invalid_pct": round(100.0 * invalid / len(probe_rows), 1),
                      "same_action_pct": round(100.0 * top / len(probe_rows), 1),
                      "example": texts[(probe_rows[0]["split"], probe_rows[0]["_idx"])]}
        print(f"{fmt:10s} 和原作者标签一致 {probe[fmt]['agree_with_label_pct']:5.1f}%  "
              f"非法 {probe[fmt]['invalid_pct']:5.1f}%  同一动作 {probe[fmt]['same_action_pct']:5.1f}%  "
              f"例：{probe[fmt]['example']!r}")
        if best_fmt is None:
            best_fmt = fmt
        else:
            b = probe[best_fmt]
            c = probe[fmt]
            if (c["agree_with_label_pct"], -c["invalid_pct"]) > (b["agree_with_label_pct"], -b["invalid_pct"]):
                best_fmt = fmt
    FORMAT = best_fmt
    if probe[FORMAT]["same_action_pct"] > MAX_SAME_PCT:
        print(f"\n⚠ 拼法可能不对：胜出的 {FORMAT} 有 {probe[FORMAT]['same_action_pct']}% 的回答是同一个动作。"
              "先别往下跑，把这一格的输出截图发到对话里。")
    del md
    if HAS_GPU:
        torch.cuda.empty_cache()
with open(os.path.join(OUT, "probe.json"), "w", encoding="utf-8") as f:
    json.dump({"chosen": FORMAT, "n": len(probe_rows), "model": MODELS_TO_RUN[0], "results": probe},
              f, ensure_ascii=False, indent=2)
print("\n用这种拼法：", FORMAT)

In [ ]:
# 第 7 格：三个模型依次在全部行上作答，每个模型写一个 preds.csv
PREDS = {}      # 模型名 → {(split, 行号): 动作}
for m in MODELS_TO_RUN:
    print(f"===== {m} =====")
    t0 = time.time()
    tk, md = load_model(m)
    check_template(tk)
    texts = generate(tk, md, ROWS, FORMAT, show_eta=True)
    run_dir = os.path.join(OUT, m)
    os.makedirs(run_dir, exist_ok=True)
    PREDS[m] = {}
    with open(os.path.join(run_dir, "preds.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["split", "idx", "pred", "raw"])
        for r in ROWS:
            key = (r["split"], r["_idx"])
            a = parse_action(texts[key], r["servers"])
            PREDS[m][key] = a
            w.writerow([r["split"], r["_idx"], a, texts[key]])
    minutes = (time.time() - t0) / 60
    config = {"name": "论文原版 " + m, "model": REF_NAME.format(m), "model_tag": "论文原版 1.5B",
              "format": FORMAT, "max_new_tokens": MAX_NEW_TOKENS, "n_per_count": N_PER_COUNT,
              "n_rows": len(ROWS), "minutes": round(minutes, 1), "torch": torch.__version__}
    with open(os.path.join(run_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    print(f"{m} 完成，用时 {minutes:.1f} 分钟，写好了 {run_dir}")
    del md
    if HAS_GPU:
        torch.cuda.empty_cache()

In [ ]:
# 第 8 格：快速看一眼（正式指标在 Mac 上用 eval.open_loop 算）
#   1. 每个模型每份测试集：准确率（= 单步最优）、非法率
#   2. 原作者干净测试集上的动作分布：本地 / 服务器 1 / 服务器 2 …（对照论文表 V）
#   3. grpo、score-grpo 和 sft 选的动作不一样的比例（三个模型是不是差不多）
summary = {"format": FORMAT, "models": {}}
for m in MODELS_TO_RUN:
    s = {}
    for split in SPLITS:
        n = 0
        hit = 0
        invalid = 0
        for r in ROWS:
            if r["split"] != split:
                continue
            n += 1
            a = PREDS[m][(split, r["_idx"])]
            if a < 0:
                invalid += 1
            elif a == r["best"]:
                hit += 1
        s[split] = {"n": n, "acc_pct": round(100.0 * hit / n, 1), "invalid_pct": round(100.0 * invalid / n, 1)}
    dist = {}
    for r in ROWS:
        if r["split"] == "friend_clean":
            a = PREDS[m][("friend_clean", r["_idx"])]
            name = "非法" if a < 0 else ("本地" if a == 0 else f"服务器{a}")
            dist[name] = dist.get(name, 0) + 1
    s["friend_clean_actions"] = dict(sorted(dist.items(), key=lambda kv: -kv[1]))
    if m != MODELS_TO_RUN[0]:
        base = MODELS_TO_RUN[0]
        diff = 0
        for key in PREDS[m]:
            if PREDS[m][key] != PREDS[base][key]:
                diff += 1
        s["differs_from_" + base + "_pct"] = round(100.0 * diff / len(PREDS[m]), 1)
    summary["models"][m] = s
    print(m, json.dumps(s, ensure_ascii=False))
with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("\n全部完成 ✓ 结果目录：", OUT)
print(sorted(os.listdir(OUT)))

## 跑完之后

1. 把第 6 格（拼法试探）和第 8 格的输出截图发到对话里。
2. 按运行文档把整个 `paper_<UTC 时间>` 文件夹下载到 Mac 的 `runs/s3/` 里。
3. 「代码执行程序 → 断开连接并删除运行时」。